# Phase 2: Automated Data Fetching & Feature Expansion

In Phase 1, we successfully trained a baseline XGBoost model. However, real-world data science requires rich, diverse features. 

In this notebook, we act as a **Data Fetcher Agent**. Instead of manually finding CSVs, we use the `yfinance` API to programmatically download new macroeconomic indicators:
- **Gold Prices (GC=F):** A traditional hedge against inflation.
- **BSE Sensex (^BSESN):** A proxy for India's economic growth and corporate health.

**Crucially, we will dynamically merge this new API data with our existing WPI and IIP data without any conflicts!**

In [ ]:
!pip install yfinance pandas

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime

# 1. Define the new features we want to add
tickers = {
    "Gold_Price_USD": "GC=F",
    "India_Stock_Market": "^BSESN"
}

print("Fetching live data from Yahoo Finance API...")
# We fetch them individually to avoid the 'database is locked' SQLite cache error in Colab
df_list = []
for name, ticker in tickers.items():
    print(f"Fetching {name}...")
    # Disabling caching issues by using a clean download
    data = yf.download(ticker, start="2014-01-01", end=datetime.today().strftime('%Y-%m-%d'), progress=False)
    if not data.empty:
        close_series = data['Close']
        close_series.name = name
        df_list.append(close_series)

# Combine individual series into one DataFrame
df_api = pd.concat(df_list, axis=1)
df_api.head()

### Data Cleaning & Resampling
Financial markets are closed on weekends, meaning there are `NaN` (missing) values. Furthermore, API data is **Daily**, but our Inflation model requires **Monthly** data. 

We must mathematically clean and aggregate this.

In [ ]:
# Forward-fill to fix missing weekend prices
df_api.ffill(inplace=True)

# Resample from Daily to Monthly ('ME' = Month End) and take the average
monthly_df = df_api.resample('ME').mean()

# Extract raw 'year' and 'month' columns 
# (These will perfectly match the raw year/month in our CSV before cyclical encoding happens!)
monthly_df['year'] = monthly_df.index.year
monthly_df['month'] = monthly_df.index.month

# Reset index to make year/month standard columns
monthly_df.reset_index(drop=True, inplace=True)

print("Successfully crunched daily data into monthly averages!")
monthly_df.head()

### The Smart Merge
Now, we load our existing `inflation_detection.csv` (which has raw WPI, IIP, Oil, and Exchange Rate up to April/May 2026). 
We use an `inner merge` to horizontally combine the new API features based on matching `year` and `month`.

**Because it is an INNER merge, the API data will automatically cut off exactly where our CSV ends (e.g., April 2026). No date conflicts!**

In [ ]:
# Load our existing Phase 1 baseline data
# Note: Make sure 'inflation_detection.csv' is uploaded to your Colab environment!
existing_data = pd.read_csv('inflation_detection.csv')

# Perform an INNER MERGE on 'year' and 'month'
vast_pipeline_data = pd.merge(existing_data, monthly_df, on=['year', 'month'], how='inner')

print(f"Merge successful! Target dates from 2014-01 to {vast_pipeline_data['year'].iloc[-1]}-{vast_pipeline_data['month'].iloc[-1]}.")
vast_pipeline_data.head()

In [ ]:
# Save the new master dataset for Phase 2 Training!
vast_pipeline_data.to_csv('v2_expanded_inflation_data.csv', index=False)
print("Saved v2_expanded_inflation_data.csv")